# LightGBM Artwork Pricer Runner

This runner is refactored for the `Data/sampled_sets` sampled CSV schema.

It trains on `Artwork_2026_finetune_40000_2023_2024_train_sample10000.csv` and evaluates on the explicit 2025 quarterly sample CSV files. Test rows are sampled per quarter and train/test overlapping lot URLs are removed from the test side to reduce leakage while preserving the requested training size.

The notebook trains two comparable LightGBM models:
- `LightGBM - No Estimates`: independent model without auction-house estimate inputs.
- `LightGBM - With Estimates`: estimate-aware model using auction-house estimate inputs.

Categorical features are passed to LightGBM as native pandas categorical columns, not one-hot encoded.

In [1]:
from pathlib import Path
import pandas as pd

from artwork_pricer.features import (
    load_artwork_data,
    split_train_test,
    make_model_frame,
    fit_feature_schema,
    apply_feature_schema,
    print_feature_summary,
)
from artwork_pricer.modeling import (
    fit_lgbm_log_price_model,
    predict_price_from_log_model,
    evaluate_price_predictions,
    evaluate_by_quarter,
    evaluate_bias_by_price_bucket,
    feature_importance_frame,
    lightgbm_training_history,
)

DATA_DIR = Path("../Data/sampled_sets_100k")
TARGET_COL = "ah_price_usd"  # use "ah_bp_price_usd" for buyer-premium-inclusive target
MIN_CATEGORY_COUNT = 20
N_MATERIAL_CLUSTERS = 50
VERBOSE_PREPROCESSING = True
TRAIN_ROWS = 40_000
TEST_ROWS_PER_QUARTER = 1000
SPLIT_RANDOM_STATE = 42
SPLIT_KEY_COL = "lot_url"
CAP_OVERSIZED_REQUESTS = True
VERBOSE_MODEL = True
LGBM_VALIDATION_FRACTION = 0.10
LGBM_EARLY_STOPPING_ROUNDS = 50
LGBM_LOG_PERIOD = 50
LGBM_EVAL_METRIC = "rmse"

MODEL_NO_ESTIMATES = "LightGBM - No Estimates"
MODEL_WITH_ESTIMATES = "LightGBM - With Estimates"

## 1. Load Data

In [2]:
raw_df = load_artwork_data(DATA_DIR, verbose=VERBOSE_PREPROCESSING)
print(raw_df.shape)
raw_df[["source_file", "source_quarter", "sale_date", "ah_price_usd", "ah_low_estimate_usd", "ah_high_estimate_usd"]].head()

[features] Loading artwork files from /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_100k
[features]   loaded Artwork_2026_finetune_100000_2022_2024_train_all_eligible.csv: 38,261 rows, 70 columns
[features]   loaded Artwork_2026_finetune_100000_q2025Q1_sample1000.csv: 1,000 rows, 70 columns
[features]   loaded Artwork_2026_finetune_100000_q2025Q2_sample1000.csv: 1,000 rows, 70 columns
[features]   loaded Artwork_2026_finetune_100000_q2025Q3_sample1000.csv: 1,000 rows, 70 columns
[features]   loaded Artwork_2026_finetune_100000_q2025Q4_sample1000.csv: 1,000 rows, 70 columns
[features] Loaded combined dataframe: 42,261 rows, 72 columns
(42261, 72)


,source_file,source_quarter,sale_date,ah_price_usd,ah_low_estimate_usd,ah_high_estimate_usd
0,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2022-07-01,31771.00,18154.86,24206.48
1,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2024-11-20,63000.00,60000.00,80000.00
2,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2023-11-23,746.65,2565.45,5130.91
3,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2022-09-23,11299.27,5663.79,7929.31
4,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2024-11-07,1852.50,3000.00,5000.00


## 2. Train/Test Split

The training set is the sampled 2023-2024 CSV; the test set is the explicit 2025 quarterly CSV files. Duplicate/leakage checks use `lot_url`.

In [3]:
train_raw, test_raw = split_train_test(
    raw_df,
    train_rows=TRAIN_ROWS,
    test_rows_per_quarter=TEST_ROWS_PER_QUARTER,
    random_state=SPLIT_RANDOM_STATE,
    dedupe_key_col=SPLIT_KEY_COL,
    cap_oversized_requests=CAP_OVERSIZED_REQUESTS,
    verbose=VERBOSE_PREPROCESSING,
)
print("train raw:", train_raw.shape)
print("test raw:", test_raw.shape)
print("test quarters:")
print(test_raw["source_quarter"].value_counts().sort_index())
print("train/test lot_url overlap:", len(set(train_raw["lot_url"].dropna()) & set(test_raw["lot_url"].dropna())))

[features] Initial split: train=38,261, explicit test=4,000
[features] Requested 40,000 train rows, only 38,261 available; using all available train rows.
[features] Using train rows: 38,261
[features] Train contains 4,001 duplicate lot_url rows; keeping them to preserve exact train size.
[features] De-duplicated test lot_url: 4,000->4,000
[features] Removed train/test overlapping lot_url values from test: 4,000->4,000
[features] Requested test rows per quarter: 1,000
[features] Actual test rows per quarter: {'2025Q1': 1000, '2025Q2': 1000, '2025Q3': 1000, '2025Q4': 1000}
[features] Final split: train=38,261, test=4,000
train raw: (38261, 72)
test raw: (4000, 72)
test quarters:
source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
Name: count, dtype: int64
train/test lot_url overlap: 0


## 3. Feature Engineering: No Estimates

This independent LightGBM model excludes auction estimates from training. Estimate fields are retained in the frame for reporting/export only.

In [4]:
train_frame, feature_cols, categorical_cols = make_model_frame(
    train_raw,
    target_col=TARGET_COL,
    include_auction_estimates=False,
    verbose=VERBOSE_PREPROCESSING,
)
test_frame, _, _ = make_model_frame(
    test_raw,
    target_col=TARGET_COL,
    include_auction_estimates=False,
    verbose=VERBOSE_PREPROCESSING,
)

schema = fit_feature_schema(
    train_frame,
    feature_cols,
    categorical_cols,
    min_category_count=MIN_CATEGORY_COUNT,
    n_material_clusters=N_MATERIAL_CLUSTERS,
    verbose=VERBOSE_PREPROCESSING,
)
X_train = apply_feature_schema(
    train_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE_PREPROCESSING
)
X_test = apply_feature_schema(
    test_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE_PREPROCESSING
)
y_train_log = train_frame["log_target_price_usd"]

print("X_train no estimates:", X_train.shape)
print("X_test no estimates:", X_test.shape)
print("categorical features:", categorical_cols)
print("numeric feature count:", len(feature_cols) - len(categorical_cols))

feature_report = print_feature_summary(train_frame, feature_cols, categorical_cols, schema=schema)
feature_report

[features] Building model frame from 38,261 rows using target_col='ah_price_usd'
[features] Filtered invalid/non-positive targets: 38,261->38,261
[features] Engineered 25 features (11 categorical, 14 numeric). include_auction_estimates=False
[features]   auction estimate fields retained for reporting but excluded from model features
[features] Building model frame from 4,000 rows using target_col='ah_price_usd'
[features] Filtered invalid/non-positive targets: 4,000->4,000
[features] Engineered 25 features (11 categorical, 14 numeric). include_auction_estimates=False
[features]   auction estimate fields retained for reporting but excluded from model features
[features] Fitting feature schema with min_category_count=20, n_material_clusters=50
[features]   fitted material clusters: 50
[features]   categorical auction_house: 7 kept levels from 5 observed
[features]   categorical location: 13 kept levels from 12 observed
[features]   categorical material_clean: 137 kept levels from 10,658 

,feature,role,transformation,missing_rate,unique_count_before_grouping,train_schema_levels,example_values
0,artist_name_grouped,categorical,Cleaned categorical from artist_name; rare art...,0.000000,9983,284.0,"Tom Wesselmann, Jacques Lipchitz, Dong Kingman"
1,artwork_title_clean,categorical,Cleaned categorical from artwork_title; rare t...,0.000000,29326,20.0,"""Stamped Indelibly"" mouth, Liseur dans un faut..."
2,auction_house,categorical,Cleaned categorical from raw auction_house; ra...,0.000000,5,7.0,"Christies, Sotheby's, Hindman"
3,dated,categorical,Cleaned categorical yes/no value from dated.,0.000000,2,4.0,"yes, no"
4,edition_clean,categorical,Cleaned categorical from edition; rare levels ...,0.000000,1,2.0,__missing__
5,location,categorical,Cleaned categorical from raw location; rare le...,0.000000,12,13.0,"London, New York, Hong Kong"
6,material_clean,categorical,Lexically cleaned raw material text; frequent ...,0.000000,10658,137.0,"coloured pencil graphite paper, cast stone, wa..."
7,material_cluster,categorical,Train-fitted TF-IDF char n-gram + MiniBatchKMe...,0.000000,50,52.0,"material_cluster_24, material_cluster_31, mate..."
8,medium,categorical,Cleaned categorical from raw medium; rare leve...,0.000000,12,12.0,"drawing, sculpture, painting"
9,signed,categorical,Cleaned categorical yes/no value from signed.,0.000000,2,4.0,"yes, no"


## 4. Feature Engineering: With Estimates

This estimate-aware LightGBM model uses the same rows and base features, plus auction-house estimate features:
- `ah_low_estimate_usd`
- `ah_high_estimate_usd`
- `auction_midpoint_usd`
- `log_auction_midpoint_usd`
- `estimate_range_ratio`

In [5]:
train_frame_est, feature_cols_est, categorical_cols_est = make_model_frame(
    train_raw,
    target_col=TARGET_COL,
    include_auction_estimates=True,
    verbose=VERBOSE_PREPROCESSING,
)
test_frame_est, _, _ = make_model_frame(
    test_raw,
    target_col=TARGET_COL,
    include_auction_estimates=True,
    verbose=VERBOSE_PREPROCESSING,
)

schema_est = fit_feature_schema(
    train_frame_est,
    feature_cols_est,
    categorical_cols_est,
    min_category_count=MIN_CATEGORY_COUNT,
    n_material_clusters=N_MATERIAL_CLUSTERS,
    verbose=VERBOSE_PREPROCESSING,
)
X_train_est = apply_feature_schema(
    train_frame_est, feature_cols_est, categorical_cols_est, schema_est, verbose=VERBOSE_PREPROCESSING
)
X_test_est = apply_feature_schema(
    test_frame_est, feature_cols_est, categorical_cols_est, schema_est, verbose=VERBOSE_PREPROCESSING
)
y_train_log_est = train_frame_est["log_target_price_usd"]

print("X_train with estimates:", X_train_est.shape)
print("X_test with estimates:", X_test_est.shape)
print("categorical features:", categorical_cols_est)
print("numeric feature count:", len(feature_cols_est) - len(categorical_cols_est))

feature_report_est = print_feature_summary(train_frame_est, feature_cols_est, categorical_cols_est, schema=schema_est)
feature_report_est

[features] Building model frame from 38,261 rows using target_col='ah_price_usd'
[features] Filtered invalid/non-positive targets: 38,261->38,261
[features] Engineered 30 features (11 categorical, 19 numeric). include_auction_estimates=True
[features]   included auction estimate numeric features: ['ah_low_estimate_usd', 'ah_high_estimate_usd', 'auction_midpoint_usd', 'log_auction_midpoint_usd', 'estimate_range_ratio']
[features] Building model frame from 4,000 rows using target_col='ah_price_usd'
[features] Filtered invalid/non-positive targets: 4,000->4,000
[features] Engineered 30 features (11 categorical, 19 numeric). include_auction_estimates=True
[features]   included auction estimate numeric features: ['ah_low_estimate_usd', 'ah_high_estimate_usd', 'auction_midpoint_usd', 'log_auction_midpoint_usd', 'estimate_range_ratio']
[features] Fitting feature schema with min_category_count=20, n_material_clusters=50
[features]   fitted material clusters: 50
[features]   categorical auction

,feature,role,transformation,missing_rate,unique_count_before_grouping,train_schema_levels,example_values
0,artist_name_grouped,categorical,Cleaned categorical from artist_name; rare art...,0.000000,9983,284.0,"Tom Wesselmann, Jacques Lipchitz, Dong Kingman"
1,artwork_title_clean,categorical,Cleaned categorical from artwork_title; rare t...,0.000000,29326,20.0,"""Stamped Indelibly"" mouth, Liseur dans un faut..."
2,auction_house,categorical,Cleaned categorical from raw auction_house; ra...,0.000000,5,7.0,"Christies, Sotheby's, Hindman"
3,dated,categorical,Cleaned categorical yes/no value from dated.,0.000000,2,4.0,"yes, no"
4,edition_clean,categorical,Cleaned categorical from edition; rare levels ...,0.000000,1,2.0,__missing__
5,location,categorical,Cleaned categorical from raw location; rare le...,0.000000,12,13.0,"London, New York, Hong Kong"
6,material_clean,categorical,Lexically cleaned raw material text; frequent ...,0.000000,10658,137.0,"coloured pencil graphite paper, cast stone, wa..."
7,material_cluster,categorical,Train-fitted TF-IDF char n-gram + MiniBatchKMe...,0.000000,50,52.0,"material_cluster_24, material_cluster_31, mate..."
8,medium,categorical,Cleaned categorical from raw medium; rare leve...,0.000000,12,12.0,"drawing, sculpture, painting"
9,signed,categorical,Cleaned categorical yes/no value from signed.,0.000000,2,4.0,"yes, no"


## 5. Train LightGBM Models

If this cell fails with `ModuleNotFoundError: lightgbm`, install LightGBM in the environment first.

In [6]:
model = fit_lgbm_log_price_model(
    X_train,
    y_train_log,
    categorical_cols=categorical_cols,
    validation_fraction=LGBM_VALIDATION_FRACTION,
    early_stopping_rounds=LGBM_EARLY_STOPPING_ROUNDS,
    log_period=LGBM_LOG_PERIOD,
    eval_metric=LGBM_EVAL_METRIC,
    random_state=SPLIT_RANDOM_STATE,
    verbose=VERBOSE_MODEL,
)
test_pred = predict_price_from_log_model(model, X_test)

model_est = fit_lgbm_log_price_model(
    X_train_est,
    y_train_log_est,
    categorical_cols=categorical_cols_est,
    validation_fraction=LGBM_VALIDATION_FRACTION,
    early_stopping_rounds=LGBM_EARLY_STOPPING_ROUNDS,
    log_period=LGBM_LOG_PERIOD,
    eval_metric=LGBM_EVAL_METRIC,
    random_state=SPLIT_RANDOM_STATE,
    verbose=VERBOSE_MODEL,
)
test_pred_est = predict_price_from_log_model(model_est, X_test_est)

training_history = lightgbm_training_history(model)
training_history.insert(0, "model", MODEL_NO_ESTIMATES)

training_history_est = lightgbm_training_history(model_est)
training_history_est.insert(0, "model", MODEL_WITH_ESTIMATES)

training_history_comparison = pd.concat([training_history, training_history_est], ignore_index=True)
training_history_comparison.tail(10)

[lightgbm model] Training LightGBM with validation diagnostics: train_fit=34,434, validation=3,827, features=25, categorical=11, eval_metric=rmse, log_period=50, early_stopping_rounds=50
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 1.41316	valid_0's l2: 1.99703
[100]	valid_0's rmse: 1.27137	valid_0's l2: 1.61639
[150]	valid_0's rmse: 1.21286	valid_0's l2: 1.47102
[200]	valid_0's rmse: 1.18399	valid_0's l2: 1.40182
[250]	valid_0's rmse: 1.16635	valid_0's l2: 1.36036
[300]	valid_0's rmse: 1.15244	valid_0's l2: 1.32812
[350]	valid_0's rmse: 1.14361	valid_0's l2: 1.30784
[400]	valid_0's rmse: 1.13474	valid_0's l2: 1.28762
[450]	valid_0's rmse: 1.12721	valid_0's l2: 1.2706
[500]	valid_0's rmse: 1.12199	valid_0's l2: 1.25886
[550]	valid_0's rmse: 1.11755	valid_0's l2: 1.24892
[600]	valid_0's rmse: 1.11369	valid_0's l2: 1.2403
[650]	valid_0's rmse: 1.10991	valid_0's l2: 1.2319
[700]	valid_0's rmse: 1.10541	valid_0's l2: 1.22194
[750]	valid_0's rmse: 1.1014

,model,dataset,metric,iteration,value
3190,LightGBM - With Estimates,valid_0,l2,791,0.329527
3191,LightGBM - With Estimates,valid_0,l2,792,0.329513
3192,LightGBM - With Estimates,valid_0,l2,793,0.329433
3193,LightGBM - With Estimates,valid_0,l2,794,0.329407
3194,LightGBM - With Estimates,valid_0,l2,795,0.329395
3195,LightGBM - With Estimates,valid_0,l2,796,0.329375
3196,LightGBM - With Estimates,valid_0,l2,797,0.329346
3197,LightGBM - With Estimates,valid_0,l2,798,0.329314
3198,LightGBM - With Estimates,valid_0,l2,799,0.329303
3199,LightGBM - With Estimates,valid_0,l2,800,0.329314


## 6. Overall Comparison Metrics

Only the approved metrics are reported: `MAE`, `RMSE`, `MAPE`, and `R2`.

In [7]:
overall_metrics = pd.DataFrame([
    evaluate_price_predictions(test_frame["target_price_usd"], test_pred, label=MODEL_NO_ESTIMATES),
    evaluate_price_predictions(test_frame_est["target_price_usd"], test_pred_est, label=MODEL_WITH_ESTIMATES),
])
overall_metrics

,model,n,MAE,RMSE,MAPE,R2
0,LightGBM - No Estimates,4000,105560.931963,690874.463249,2.596462,0.050108
1,LightGBM - With Estimates,4000,45881.846757,283889.411435,0.950060,0.839611


## 7. Quarterly Comparison Metrics

In [8]:
quarter_metrics = pd.concat(
    [
        evaluate_by_quarter(test_frame, test_pred, label=MODEL_NO_ESTIMATES),
        evaluate_by_quarter(test_frame_est, test_pred_est, label=MODEL_WITH_ESTIMATES),
    ],
    ignore_index=True,
).sort_values(["quarter", "model"]).reset_index(drop=True)
quarter_metrics

,model,n,MAE,RMSE,MAPE,R2,quarter
0,LightGBM - No Estimates,1000,63445.491008,377494.593751,2.171498,0.160695,2025Q1
1,LightGBM - With Estimates,1000,30501.585825,180946.128770,1.007927,0.807160,2025Q1
2,LightGBM - No Estimates,1000,168105.316063,978667.351722,2.672428,0.036220,2025Q2
3,LightGBM - With Estimates,1000,73935.089940,405312.196688,1.061352,0.834695,2025Q2
4,LightGBM - No Estimates,1000,30773.831685,171165.536744,2.612508,0.097303,2025Q3
5,LightGBM - With Estimates,1000,13757.918156,55255.639315,0.972065,0.905927,2025Q3
6,LightGBM - No Estimates,1000,159919.089095,882972.537101,2.929414,0.021404,2025Q4
7,LightGBM - With Estimates,1000,65332.793107,349714.351610,0.758897,0.846490,2025Q4


## 8. Metrics By Price Range

In [9]:
bucket_metrics = pd.concat(
    [
        evaluate_bias_by_price_bucket(test_frame, test_pred, label=MODEL_NO_ESTIMATES),
        evaluate_bias_by_price_bucket(test_frame_est, test_pred_est, label=MODEL_WITH_ESTIMATES),
    ],
    ignore_index=True,
).sort_values(["price_bucket", "model"]).reset_index(drop=True)
bucket_metrics

,model,n,MAE,RMSE,MAPE,R2,price_bucket
0,LightGBM - No Estimates,302,1.595741e+05,1.917626e+05,0.721874,-2.282294,$100k-$500k
1,LightGBM - With Estimates,302,8.871826e+04,1.218381e+05,0.424657,-0.325000,$100k-$500k
2,LightGBM - No Estimates,1256,6.144477e+03,1.672063e+04,2.799420,-222.622766,$1k-$5k
3,LightGBM - With Estimates,1256,2.325408e+03,8.771433e+03,0.969267,-60.539158,$1k-$5k
4,LightGBM - No Estimates,633,3.545137e+04,5.038388e+04,0.738675,-5.281663,$25k-$100k
5,LightGBM - With Estimates,633,2.180380e+04,2.885289e+04,0.452930,-1.060014,$25k-$100k
6,LightGBM - No Estimates,163,2.011326e+06,3.410037e+06,0.825548,-0.574305,$500k+
7,LightGBM - With Estimates,163,8.019461e+05,1.394456e+06,0.372180,0.736743,$500k+
8,LightGBM - No Estimates,1252,1.134975e+04,2.382393e+04,1.047288,-18.813342,$5k-$25k
9,LightGBM - With Estimates,1252,6.993738e+03,1.571052e+04,0.636071,-7.616126,$5k-$25k


## 9. Feature Importance

In [10]:
importance = feature_importance_frame(model, feature_cols)
importance.insert(0, "model", MODEL_NO_ESTIMATES)

importance_est = feature_importance_frame(model_est, feature_cols_est)
importance_est.insert(0, "model", MODEL_WITH_ESTIMATES)

importance_comparison = pd.concat([importance, importance_est], ignore_index=True)
importance_comparison.head(40)

,model,feature,split_importance,gain_importance
0,LightGBM - No Estimates,artist_name_grouped,4186,5.811971e+05
1,LightGBM - No Estimates,medium,1150,2.723018e+05
2,LightGBM - No Estimates,auction_house,819,2.219935e+05
3,LightGBM - No Estimates,location,1914,1.919015e+05
4,LightGBM - No Estimates,material_cluster,4761,1.843429e+05
5,LightGBM - No Estimates,height_cm,5466,1.637057e+05
6,LightGBM - No Estimates,sale_date_ordinal,6332,1.365966e+05
7,LightGBM - No Estimates,width_cm,4818,8.741567e+04
8,LightGBM - No Estimates,year_of_creation_num,5382,8.130686e+04
9,LightGBM - No Estimates,material_clean,1922,5.819802e+04


## 10. Export Results

In [11]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

scored_test = test_frame[[
    "ID",
    "lot_url",
    "source_quarter",
    "sale_date",
    "target_price_usd",
    "ah_low_estimate_usd",
    "ah_high_estimate_usd",
    "auction_midpoint_usd",
    "log_auction_midpoint_usd",
    "estimate_range_ratio",
]].copy()
scored_test["lightgbm_pred_no_estimates_usd"] = test_pred
scored_test["lightgbm_error_no_estimates_usd"] = scored_test["lightgbm_pred_no_estimates_usd"] - scored_test["target_price_usd"]
scored_test["lightgbm_pred_with_estimates_usd"] = test_pred_est
scored_test["lightgbm_error_with_estimates_usd"] = scored_test["lightgbm_pred_with_estimates_usd"] - scored_test["target_price_usd"]

scored_test.to_csv(output_dir / "test_predictions.csv", index=False)
overall_metrics.to_csv(output_dir / "overall_metrics.csv", index=False)
quarter_metrics.to_csv(output_dir / "quarter_metrics.csv", index=False)
bucket_metrics.to_csv(output_dir / "price_bucket_metrics.csv", index=False)
importance_comparison.to_csv(output_dir / "feature_importance.csv", index=False)
training_history_comparison.to_csv(output_dir / "training_history.csv", index=False)
feature_report.to_csv(output_dir / "feature_report_no_estimates.csv", index=False)
feature_report_est.to_csv(output_dir / "feature_report_with_estimates.csv", index=False)

print("Wrote outputs to", output_dir.resolve())

Wrote outputs to /srv/notebooks/[6.0] ML Pricers/LightGBM_Refactored/outputs
